# Creating Ground Truth

<a target="_blank" href="https://colab.research.google.com/github/zentralbibliothek-zuerich/zblab-summerschool-2026/blob/main/notebooks/05_annotate_labels.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook contains boilerplate code that you don't need to interact with directly.

The sections where you can safely experiment or customize are clearly marked with such comments:

```python
# ⬇️⬇️⬇️
YOUR_INPUT = ""
# ⬆️⬆️⬆️

## Setup

### Housekeeping (no interaction required)

In [ ]:
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm # from tqdm.notebook import tqdm for better display in notebooks

tqdm.pandas()

In [ ]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBLabSummerSchool2026/data') if IN_COLAB else Path('../data')

In [ ]:
def confirm(question: str = "Do you want to execute this cell?") -> bool:
    """Ask for a yes/no confirmation in the notebook.

    Args:
        question: Prompt shown to the user.

    Returns:
        True for yes, False for no.
    """
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

### Setup (interaction required)

In [ ]:
### ⬇️⬇️⬇️ Adjust here if you want to continue with your own query
CORPUS_NAME = "armenpflege"
USE_YOUR_DATA = False # Set to True if you want to use your own data
### ⬆️⬆️⬆️

In [ ]:
if IN_COLAB and USE_YOUR_DATA: # and confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive, files
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

## Load the data

### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [ ]:
if USE_YOUR_DATA:
    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module
    raw_df = pd.read_parquet(RAWDATA_PATH)

    SENTENCES_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.sentences.parquet" # Use data from filtering module
    sentences_df = pd.read_parquet(SENTENCES_PATH)

    raw_df = raw_df.join(sentences_df, on="id")

### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load from example

In [ ]:
if not USE_YOUR_DATA:
    RAWDATA_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.parquet"
    print(f"Loading raw data ...", end=" ")
    raw_df = pd.read_parquet(RAWDATA_URL)
    print("Done!")

    SENTENCES_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.sentences.parquet"
    print(f"Loading sentences data ...", end=" ")
    sentences_df = pd.read_parquet(SENTENCES_URL)
    print("Done!")

    raw_df = raw_df.join(sentences_df)

## Create pseudo-paragraphs to label

For this example, we want to label the *articles* where the search terms are occuring, as opposed to e.g. the singular *mentions* of the search term.
To give us more context during labeling, we extract 3 sentences before and 3 sentences after a sentence with a search term.
We call these 7-sentence chunks *pseudo-paragraphs*.

When the LLM is doing the labeling task, we will also only show it these pseudo paragraphs as a *feature*, for three reasons:
1. It is cheaper than sending whole pages.
2. It is more precise. Impresso mainly returned physical *pages*, and not thematic *articles*. A lot of the text has little to do with our search terms.
3. It is simpler. For full-fledged research, you might consider giving the LLM additional context, such as more text or the year of publication. ⚠️ Consider however that this might introduce biases!

This procedure simplifies the task considerably and has some rough edges:
- Pseudo-paragraphs don't necessarily give enough context to determine the nature of an occurence. We can look at the full text, but the LLM will only see the pseudo-paragraphs.

In [ ]:
### ⬇️⬇️⬇️ Adjust here
SEARCH_TERMS = ["armenpflege"] #["armensteuer", "armensteuern", "armenausgaben", "armengut", "armengüter"]
### ⬆️⬆️⬆️

In [ ]:
N_BEFORE = 3 # Number of sentences before the sentence containing the search term to include in the pseudo-paragraph
N_AFTER = 3 # Number of sentences after the sentence containing the search term to include in the pseudo-paragraph

from itertools import groupby
from operator import itemgetter

def create_pseudo_paragraphs(sentences: list[str], search_terms: list[str]) -> str:
    """Create pseudo-paragraphs of the articles based on the search terms.

    An article might contain the search terms multiple times. In these cases, the pseudo-paragraphs
    are combined.
    
    Args:
        sentences (list[str]): List of sentences to search within.
        search_terms (list[str]): List of search terms to look for in the sentences.

    Returns:
        str: A string containing the pseudo-paragraphs joined by "   [...]   ".
    """
    search_terms = [term.lower() for term in search_terms]

    hit_indices = [i for i, sentence in enumerate(sentences) if any(term in sentence.lower() for term in search_terms)]
    pseudo_paragraph_indices = []
    for i in hit_indices:
        start = max(0, i - N_BEFORE)
        end = min(len(sentences), i + N_AFTER + 1)
        pseudo_paragraph_indices.extend(range(start, end))
    pseudo_paragraph_indices = sorted(set(pseudo_paragraph_indices))
    
    # Join all consecutive sentences with " "
    pseudo_paragraphs = []
    for _, group in groupby(enumerate(pseudo_paragraph_indices), lambda x: x[0] - x[1]):
        indices = list(map(itemgetter(1), group))
        pseudo_paragraphs.append(" ".join(sentences[i] for i in indices))
    
    return  "   [...]   ".join(pseudo_paragraphs)

raw_df["pseudo_paragraph"] = raw_df["_sentences"].progress_apply(lambda s: create_pseudo_paragraphs(s, SEARCH_TERMS))

In [ ]:
p_df = raw_df.copy()

# Only keep the columns we need for annotation
p_df = p_df[["date", "media_title", "pseudo_paragraph"]]

# Drop rows where pseudo_paragraph is empty or NaN (i.e. no search term was found)
p_df = p_df.dropna(subset=["pseudo_paragraph"])
p_df = p_df[p_df["pseudo_paragraph"] != ""]

p_df

### Randomize the data for labeling

We want to label a random sample of our data, so our evaluation of the LLM labeler in the next step is more likely to generalize to the whole dataset.

In [ ]:
p_df = p_df.sample(frac=1, random_state=42)

### Add helper columns

#### URL

We add a column "url" which directly opens the article in the Impresso viewer.

This can be helpful to interpret the pseudo-paragraphs

**But remember:** The LLM will not be able to use this when it labels the data!

In [ ]:
def parse_ids_to_urls(id: str) -> str:
    """Build an Impresso viewer URL from an article ID.

    Args:
        id: Impresso article identifier.

    Returns:
        URL that opens the article in the web viewer.
    """
    id_parts = id.split("-")
    article_id = id_parts[-1]
    issue_id = "-".join(id_parts[:-1])
    return f"https://impresso-project.ch/app/issue/{issue_id}/view?p=5&articleId={article_id}"

p_df["url"] = p_df.index.to_series().apply(parse_ids_to_urls)

# Make the url column the first column
cols = p_df.columns.tolist()
cols = cols[-1:] + cols[:-1]
p_df = p_df[cols]

#### Label

An empty `"label"` column is added for entering a label.

In [ ]:
p_df["label"] = ""

## Download

Save the pseudo-paragraphs and download the file.

Then look at your data, work on a label scheme and start annotating.

In [ ]:
if USE_YOUR_DATA: 
    p_df.to_csv(DATA_DIR / f"{CORPUS_NAME}.filtered.pp.csv")
    if IN_COLAB:
        files.download(str(DATA_DIR / f"{CORPUS_NAME}.filtered.pp.csv"))
else:
    print("You can download the pseudo-paragraphs data for the 'Armenpflege' corpus from the following URL:")
    PP_DATA_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.pp.csv"
    print(PP_DATA_URL)

If you are annotating the data in Excel, you can use the following VBA code to set your search terms in bold for easier reading.

```vb
Sub BoldOccurrencesInSelection()
    Dim searchText As String
    Dim cell As Range
    Dim startPos As Long
    Dim textLen As Long
    Dim count As Long

    ' Check something is selected
    If Selection Is Nothing Then
        MsgBox "No cells selected."
        Exit Sub
    End If

    ' Prompt user for the search string
    searchText = InputBox("Enter the text to make bold:", "Bold Text in Selection")

    If searchText = "" Then
        MsgBox "No text entered. Macro cancelled."
        Exit Sub
    End If

    textLen = Len(searchText)
    count = 0

    ' Loop through selected cells only
    For Each cell In Selection
        If cell.HasFormula = False And InStr(1, cell.Value, searchText, vbTextCompare) > 0 Then
            startPos = 1
            Do
                startPos = InStr(startPos, cell.Value, searchText, vbTextCompare)
                If startPos = 0 Then Exit Do
                cell.Characters(startPos, textLen).Font.Bold = True
                count = count + 1
                startPos = startPos + textLen
            Loop
        End If
    Next cell

    MsgBox count & " occurrence(s) of """ & searchText & """ made bold in selection."
End Sub
```

## Annotate

For our 'Armenpflege' example, we inductively arrived at three classes for the articles.

![Labeling scheme for armenpflege-articles](../assets/armenpflege_kategoriensystem.excalidraw.png)

| Label                    | Description                                                                                                                                                                                                                                     |
| ------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| *Meinungsartikel*        | An article where someone (the author or a cited speaker) explicitely mentions how the 'Armenpflege' or a related topic ought to be or where an explicit valorisation is made.                                                                   |
| *Unspezifischer Bericht* | An article containing a recollection of events or a description of facts without any explicit markers of subjectivity. Does not include articles that fall under "Einzelfalldarstellung".                                                       |
| *Einzelfalldarstellung*  | An article that describes the circumstances of a concrete person or group that interacted with the 'Armenpflege', mainly as recipients. Articles around singular politicians or institutional actors do not fall under 'Einzelfalldarstellung'. |

These categories are partially driven by the hypothesis that the share of 'Meinungsartikel' becomes smaller and smaller as time progresses and the institution of "Armenpflege" changes from novelty to normalcy.

This annotation scheme is problematic for different reasons:
- The categories belong to different dimensions; the presentation of a singular case can be part of an opinion piece or not. The scheme constructs a world in which they are exclusive categories.
- The prevalence of the categories is imbalanced.
- An opinion piece can be against the 'Armenpflege' in general, or against a very specific aspect of it.


## After annotating

When working with your own data, save your labeled articles on your Google Drive in the Summerschool folder.
The file should have this name:

`{CORPUS_NAME}.filtered.pp.label.{YOUR_NAME}.csv`

To export the File as `csv`, you need to explicitly change the filetype during saving.

⚠️ You need to choose **CSV UTF-8**, and not any other csv-dialect!

![Save table as csv in Excel](../assets/save_excel_as_csv.png)